In [17]:
#importing everything needed
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

#Setting up some additional display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [18]:
# calling to dataset
X_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")
X_val = pd.read_csv("X_val.csv")
y_val = pd.read_csv("y_val.csv")
X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv")

# recombine for EDA purposes
train_data = X_train.copy()
train_data['carbon_emissions'] = y_train

print("Train Shape:", train_data.shape)
display(train_data.head())
display(train_data.info())

Train Shape: (1517, 14)


,Country Name,Year,"Agriculture, forestry, and fishing, value added (% of GDP)",Electric power consumption (kWh per capita),Electricity production from hydroelectric sources (% of total),Electricity production from natural gas sources (% of total),Electricity production from nuclear sources (% of total),"Energy imports, net (% of energy use)",Fossil fuel energy consumption (% of total),"Industry (including construction), value added (% of GDP)",Renewable energy consumption (% of total final energy consumption),Trade (% of GDP),Urban population (% of total population),carbon_emissions
0,Algeria,2001,8.8862,711.8139,0.2592,96.8300,0.0000,-406.3560,99.7578,40.8146,0.4000,54.5296,60.6177,2.7392
1,Algeria,2002,8.4130,733.2091,0.2062,97.6345,0.0000,-391.5575,99.7229,39.7667,0.5000,56.5896,61.3813,2.8247
2,Algeria,2003,8.4567,788.5736,0.8961,96.7806,0.0000,-401.8576,99.7135,42.0401,0.5000,57.7691,62.1428,2.9993
3,Algeria,2004,7.7222,805.6813,0.8032,96.9984,0.0000,-405.2957,99.7206,44.9815,0.4000,61.3580,62.9023,2.9614
4,Algeria,2005,6.5593,891.7146,1.6364,96.2494,0.0000,-412.4430,99.5923,48.5068,0.6000,66.8405,63.6599,3.0626


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1517 entries, 0 to 1516
Data columns (total 14 columns):
 #   Column                                                              Non-Null Count  Dtype  
---  ------                                                              --------------  -----  
 0   Country Name                                                        1517 non-null   object 
 1   Year                                                                1517 non-null   int64  
 2   Agriculture, forestry, and fishing, value added (% of GDP)          1517 non-null   float64
 3   Electric power consumption (kWh per capita)                         1517 non-null   float64
 4   Electricity production from hydroelectric sources (% of total)      1517 non-null   float64
 5   Electricity production from natural gas sources (% of total)        1517 non-null   float64
 6   Electricity production from nuclear sources (% of total)            1517 non-null   float64
 7   Energy imports,

None

In [19]:
# defining target variable and features
target = "carbon_emissions"

X_train_model = X_train.drop(columns=['Country Name', 'Year'])
X_val_model = X_val.drop(columns=['Country Name', 'Year'])
X_test_model = X_test.drop(columns=['Country Name', 'Year'])

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_model)
X_val_scaled = scaler.transform(X_val_model)
X_test_scaled = scaler.transform(X_test_model)

print("Feature columns:")
print(X_train_model.columns.tolist())
print("\nTarget column:")
print(target)

Feature columns:
['Agriculture, forestry, and fishing, value added (% of GDP)', 'Electric power consumption (kWh per capita)', 'Electricity production from hydroelectric sources (% of total)', 'Electricity production from natural gas sources (% of total)', 'Electricity production from nuclear sources (% of total)', 'Energy imports, net (% of energy use)', 'Fossil fuel energy consumption (% of total)', 'Industry (including construction), value added (% of GDP)', 'Renewable energy consumption (% of total final energy consumption)', 'Trade (% of GDP)', 'Urban population (% of total population)']

Target column:
carbon_emissions


In [20]:
# identifying which columns are numerical and categorical
categorical_features = X_train_model.select_dtypes(include=["object"]).columns.tolist()
numeric_features = X_train_model.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

Categorical features: []
Numeric features: ['Agriculture, forestry, and fishing, value added (% of GDP)', 'Electric power consumption (kWh per capita)', 'Electricity production from hydroelectric sources (% of total)', 'Electricity production from natural gas sources (% of total)', 'Electricity production from nuclear sources (% of total)', 'Energy imports, net (% of energy use)', 'Fossil fuel energy consumption (% of total)', 'Industry (including construction), value added (% of GDP)', 'Renewable energy consumption (% of total final energy consumption)', 'Trade (% of GDP)', 'Urban population (% of total population)']


In [21]:
# created a helper function to evalute the models we choose
def evaluate_model(name, model, X_train, X_val, X_test, y_train, y_val, y_test):
    model.fit(X_train, y_train.values.ravel())

    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    results = {
        "Model": name,
        "Train R2": r2_score(y_train, y_train_pred),
        "Val R2": r2_score(y_val, y_val_pred),
        "Test R2": r2_score(y_test, y_test_pred),
        "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
        "Val RMSE": np.sqrt(mean_squared_error(y_val, y_val_pred)),
        "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),
        "Train MAE": mean_absolute_error(y_train, y_train_pred),
        "Val MAE": mean_absolute_error(y_val, y_val_pred),
        "Test MAE": mean_absolute_error(y_test, y_test_pred)
    }

    return results

In [22]:
# Linear Regression Baseline
linear_model = LinearRegression()

linear_results = evaluate_model(
    "Linear Regression",
    linear_model,
    X_train_model, X_val_model, X_test_model,
    y_train, y_val, y_test
)

linear_results

{'Model': 'Linear Regression',
 'Train R2': 0.7111802812115555,
 'Val R2': 0.8035395422022499,
 'Test R2': 0.7525496103019362,
 'Train RMSE': np.float64(3.167706686262153),
 'Val RMSE': np.float64(2.8744416308583376),
 'Test RMSE': np.float64(2.072817240620446),
 'Train MAE': 2.0451934226083184,
 'Val MAE': 1.8711763210704393,
 'Test MAE': 1.6550869406258681}

In [23]:
#Based on the results we can see that the R2 for both train and test are not only extremely good but also very similar.
#This may be caused by on hot coding countries, which might be improving the overall model. The closeness between both 
#the training and testing does suggest limited overfitting which is good for the model. 

In [24]:
# Ridge Regression
ridge_alphas = np.logspace(-3, 6, 100)

ridge_model = RidgeCV(alphas=ridge_alphas, cv=5)

ridge_results = evaluate_model(
    "Ridge Regression",
    ridge_model,
    X_train_scaled, X_val_scaled, X_test_scaled,
    y_train, y_val, y_test
)

ridge_results

{'Model': 'Ridge Regression',
 'Train R2': 0.6823172588267248,
 'Val R2': 0.7947784438357183,
 'Test R2': 0.72938247670544,
 'Train RMSE': np.float64(3.3222197187058082),
 'Val RMSE': np.float64(2.9378350409234684),
 'Test RMSE': np.float64(2.1676786427339514),
 'Train MAE': 2.20533606871106,
 'Val MAE': 1.8716716983154809,
 'Test MAE': 1.6052608244232662}

In [25]:
# Finding the best alpha for Ridge
best_ridge_alpha = ridge_model.alpha_
print("Best Ridge alpha:", best_ridge_alpha)

Best Ridge alpha: 351.11917342151344


In [26]:
#Ridge Regression created nearly identical results to the baseline model. Since the optimal regularization 
#strength was very small, the Ridge model did not really improve performance, meaning that coefficient shrinkage
#was not really necessary for this dataset.

In [27]:
# Lasso Regression
lasso_alphas = np.logspace(-6, 6, 100)

lasso_model = LassoCV(alphas=lasso_alphas, cv=5, random_state=42, max_iter=20000)

lasso_results = evaluate_model(
    "Lasso Regression",
    lasso_model,
    X_train_scaled, X_val_scaled, X_test_scaled,
    y_train, y_val, y_test
)

lasso_results



{'Model': 'Lasso Regression',
 'Train R2': 0.6977891197197736,
 'Val R2': 0.7901365614808571,
 'Test R2': 0.8308268299403566,
 'Train RMSE': np.float64(3.2403102069941347),
 'Val RMSE': np.float64(2.9708745299625168),
 'Test RMSE': np.float64(1.7138889742520125),
 'Train MAE': 2.0045422249949985,
 'Val MAE': 1.8240521825113964,
 'Test MAE': 1.2886724975428567}

In [28]:
print("Best Lasso alpha:", lasso_model.alpha_)

feature_coefficients = pd.DataFrame({
    'Feature': X_train_model.columns,
    'Coefficient': lasso_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print(feature_coefficients)

Best Lasso alpha: 0.2848035868435805
                                              Feature  Coefficient
1         Electric power consumption (kWh per capita)       3.8949
2   Electricity production from hydroelectric sour...      -0.9568
7   Industry (including construction), value added...       0.7468
8   Renewable energy consumption (% of total final...      -0.7261
3   Electricity production from natural gas source...       0.1627
4   Electricity production from nuclear sources (%...      -0.1421
0   Agriculture, forestry, and fishing, value adde...      -0.0249
6         Fossil fuel energy consumption (% of total)       0.0000
5               Energy imports, net (% of energy use)      -0.0000
9                                    Trade (% of GDP)       0.0000
10           Urban population (% of total population)       0.0000


In [29]:
# Lasso Regression performed better than both Ridge and Linear Regression.

In [30]:
# =========================================
# 11. Compare results
# =========================================
results_df = pd.DataFrame([linear_results, ridge_results, lasso_results])
display(results_df)

,Model,Train R2,Val R2,Test R2,Train RMSE,Val RMSE,Test RMSE,Train MAE,Val MAE,Test MAE
0,Linear Regression,0.7112,0.8035,0.7525,3.1677,2.8744,2.0728,2.0452,1.8712,1.6551
1,Ridge Regression,0.6823,0.7948,0.7294,3.3222,2.9378,2.1677,2.2053,1.8717,1.6053
2,Lasso Regression,0.6978,0.7901,0.8308,3.2403,2.9709,1.7139,2.0045,1.8241,1.2887


In [31]:
# Overall, all three models performed closely to each other, with some improvement from regularization for Lasso. 
#This tells us that the dataset already supports a strong linear fit, but some of the predictive 
#power may come from country level identity effects rather than only from the economical development indicators.

In [32]:
# Showing Lasso selected features
coef_df = pd.DataFrame({
    "Feature": X_train_model.columns,
    "Coefficient": lasso_model.coef_
})

selected_features = coef_df[coef_df["Coefficient"] != 0].sort_values(
    by="Coefficient", key=np.abs, ascending=False
)

print("Number of features selected by Lasso:", selected_features.shape[0])
display(selected_features)

Number of features selected by Lasso: 7


,Feature,Coefficient
1,Electric power consumption (kWh per capita),3.8949
2,Electricity production from hydroelectric sour...,-0.9568
7,"Industry (including construction), value added...",0.7468
8,Renewable energy consumption (% of total final...,-0.7261
3,Electricity production from natural gas source...,0.1627
4,Electricity production from nuclear sources (%...,-0.1421
0,"Agriculture, forestry, and fishing, value adde...",-0.0249
